# Demo: Branching Skeleton

In [1]:
from mascaf import *

import logging

logging.basicConfig(level=logging.INFO)

In [2]:
mm = MeshManager(mesh_path="../../data/demo/test_branching_3.obj")
mm.print_mesh_analysis()
raw_skeleton = SkeletonGraph.from_txt(f"../../data/demo/test_branching_3.polylines.txt")
# raw_skeleton.prune_short_branches_inplace(min_length_fraction=1)
mm.visualize_mesh_3d(title="Branching test model", skel=raw_skeleton)

INFO:mascaf.mesh:Loaded mesh: 1417 vertices, 2830 faces
INFO:mascaf.mesh:Mesh Analysis Report
INFO:mascaf.mesh:====================
INFO:mascaf.mesh:
Geometry:
INFO:mascaf.mesh:  * Vertices: 1417
INFO:mascaf.mesh:  * Faces: 2830
INFO:mascaf.mesh:  * Components: 1
INFO:mascaf.mesh:  * Volume: 5.11
INFO:mascaf.mesh:  * Bounds: [-1.0, -1.3, -3.6] to [1.0, 1.4, 1.0]
INFO:mascaf.mesh:
Mesh Quality:
INFO:mascaf.mesh:  * Watertight: True
INFO:mascaf.mesh:  * Winding Consistent: True
INFO:mascaf.mesh:  * Normal Direction: outward
INFO:mascaf.mesh:  * Duplicate Vertices: 0
INFO:mascaf.mesh:  * Degenerate Faces: 0
INFO:mascaf.mesh:
Topology:
INFO:mascaf.mesh:  * Genus: 0
INFO:mascaf.mesh:  * Euler Characteristic: 2
INFO:mascaf.mesh:
No issues detected
INFO:mascaf.mesh:
Recommendation:
INFO:mascaf.mesh:  Mesh appears to be in good condition.
INFO:mascaf.mesh:====================


In [3]:
optimizer_options = BasisOptimizerOptions(
    do_pruning=False,
    do_snapping=True,
    do_forcing=True,
    max_iterations=5,
    step_size=0.01,
    smoothing_weight=0.1,
    preserve_terminal_nodes=False,
    preserve_branch_nodes=False,
)
skeleton = raw_skeleton

INFO:mascaf.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mascaf.skeleton_optimizer:Starting skeleton optimization...
INFO:mascaf.skeleton_optimizer:  Nodes: 137
INFO:mascaf.skeleton_optimizer:  Max iterations: 5
INFO:mascaf.skeleton_optimizer:  Step size: 0.0100
INFO:mascaf.skeleton_optimizer:  Smoothing weight: 0.1000



Optimizing skeleton (serial)...


INFO:mascaf.skeleton_optimizer:  Iteration 0: avg movement = 0.009481
INFO:mascaf.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mascaf.skeleton_optimizer:Optimization complete


In [10]:
max_edge_length = 1.0

swc_filepath = "../../data/demo/test_branching_3.swc"

radius_strategy = "equivalent_area"
print(f"Computing skeleton for radius_strategy={radius_strategy} ...", end="")
morph = CableFitter(
    FitOptions(
        max_edge_length=max_edge_length,
        radius_strategy=radius_strategy,
        basis_optimizer_options=optimizer_options,
    )
).fit(
    mm.mesh,
    skeleton,
)
# write swc to file
morph.to_swc_file(swc_filepath)
# validation
validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:mascaf.graph_fitting:Tracing done: nodes=9, edges=8, samples=13, section=0, fallback=0 (0.0%)


Computing skeleton for radius_strategy=equivalent_area ...

INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 9 nodes, 8 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 3.6010
INFO:mascaf.validation:---- Ratio:             0.7051
INFO:mascaf.validation:---- Error:             -1.5061
INFO:mascaf.validation:---- Relative error:    -29.49%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   18.8415
INFO:mascaf.validation:---- Ratio:             0.8442
INFO:mascaf.validation:---- Error:             -3.4761
INFO:mascaf.validation:---- Relative error:    -15.58%
INFO:mascaf.validation:Validation Results, accou

In [12]:
# plot using swctools
from swctools import SWCModel, plot_model

model = SWCModel.from_swc_file(swc_filepath)
model.print_attributes(node_info=False, edge_info=False)
title = f"Branching Skeleton"
fig = plot_model(
    swc_model=model,
    title=title,
    plot_endcaps=True,
    hide_axes=True,
    centroid_color="black",
    centroid_line_width=5,
    opacity=0.5,
)
fig.show()

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=9 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_parse_result records=9 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_swc_file built nodes=9 edges=8 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=8 sides=16 end_caps=False verts=256 faces=256
INFO:swctools.geometry:FrustaSet.from_swc_model edges=8 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=8 show_frusta=True show_centroid=True


SWCModel: nodes=9, edges=8, components=1, cycles=0, branch_points=2, roots=1, leaves=3, self_loops=0, density=0.2222


In [15]:
# normalize radii to match mesh surface area
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)

# save normalized to file
swc_filepath_normalized = "../../data/demo/test_branching_3_normalized.swc"
morph.to_swc_file(swc_filepath_normalized)

# load and plot
swc_model = SWCModel.from_swc_file(swc_filepath_normalized)
fig = plot_model(
    swc_model=swc_model,
    slider=False,
    title="",
    hide_axes=True,
    centroid_color="black",
    centroid_line_width=5,
    opacity=0.5,
    plot_endcaps=True,
)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=9 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_parse_result records=9 reconnections=0 header=4
INFO:swctools.model:SWCModel.from_swc_file built nodes=9 edges=8 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=8 sides=16 end_caps=False verts=256 faces=256
INFO:swctools.geometry:FrustaSet.from_swc_model edges=8 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=8 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 9 nodes, 8 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 4.6834
INFO:mascaf.validation:---- Ratio:             0.9170
INFO:mascaf.validation:---- Error:             -0.4237
INFO:mascaf.validation:---- Relative error:    -8.30%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   22.3177
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- Error:             -0.0000
INFO:mascaf.validation:---- Relative error:    -0.00%
INFO:mascaf.validation:Validation Results, account